<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/TextSentimentAnalysisRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sayın Hocam,

Ödevi incelerken aşağıdaki hususları dikkate almanızı rica ederim.

Kaggle veri indirme ve yükleme süreçlerinde, veri boyutunun yüksek olması nedeniyle doğrudan klasör olarak indirme yapılamamıştır. Bu nedenle veri, Kaggle API kullanılarak .json formatında indirilmiş ve Colab ortamına yüklenmiştir. Colab üzerinde veri erişimi için ilgili kaggle.json dosyasının yolu tanımlanarak API yetkilendirmesi gerçekleştirilmiştir. Uygulamanın çalıştırılabilmesi için önceden KAGGLE_USERNAME ve KAGGLE_KEY değişkenlerinin tanımlanması gerekmektedir.

Bilgilerinize sunarım.

Saygılarımla

In [ ]:
from google.colab import userdata
import os
import zipfile

In [ ]:
!pip install kaggle

Veri Seti ve Sınıf Seçimi

In [ ]:
from google.colab import userdata
import os

print("Colab Secrets'tan Kaggle API bilgileri okunuyor...")
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("API bilgileri başarıyla okundu.")
except userdata.SecretNotFoundError:
    print("\nHata: 'KAGGLE_USERNAME' veya 'KAGGLE_KEY' gizli anahtarları bulunamadı.")
    print("Lütfen yukarıdaki adımları takip ederek anahtarları doğru şekilde eklediğinizden emin olun.")
except Exception as e:
    print(f"\nBeklenmeyen bir hata oluştu: {e}")


In [ ]:
# IMDB lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

zip_file_name = 'imdb-dataset-of-50k-movie-reviews.zip'
extract_dir = 'imdb_dataset'

if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

# ZIP dosyasını çıkar
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Veri seti '{zip_file_name}' dosyasından '{extract_dir}' dizinine çıkarıldı.")

# Çıkarılan dosyaları listele (isteğe bağlı)
print("\nÇıkarılan dosyalar:")
!ls {extract_dir}

In [ ]:
import pandas as pd
df = pd.read_csv(f'{extract_dir}/IMDB Dataset.csv')

print("\nIMDB veri setinin ilk 5 satırı:")
display(df.head())

## Sentiment Analizi için RNN Modeli Oluşturma Adımları

1.  **Veri Ön İşleme (Preprocessing)**:
    *   **Metin Temizleme**
    *   **Tokenizasyon**
    *   **Kelime Dizini Oluşturma**
    *   **Sıraları Uzunluklara Sabitleme (Padding)**
    *   **Etiketleri Sayısallaştırma**

2.  **Veriyi Ayırma**

3.  **Model Oluşturma (RNN)**:
    *   **Embedding Katmanı**
    *   **RNN Katmanı (LSTM/GRU)**

4.  **Model Eğitimi**

5.  **Model Değerlendirme**

### 1. Veri Ön İşleme

In [ ]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Metin temizleme fonksiyonu
def clean_text(text):
    # HTML etiketlerini kaldır
    text = re.sub(r'<.*?>', '', text)
    # Özel karakterleri ve sayıları kaldır
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Küçük harfe dönüştür
    text = text.lower()
    return text

df['clean_review'] = df['review'].apply(clean_text)

print("Temizlenmiş verinin ilk 5 satırı:")
display(df[['review', 'clean_review']].head())


In [ ]:
# Etiketleri sayısallaştırma
label_encoder = LabelEncoder()
df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

print("Etiketlerin kodlanmış hali:", df['sentiment_encoded'].unique())
print("Etiket eşleşmesi:", list(label_encoder.classes_), list(label_encoder.transform(label_encoder.classes_)))

max_words = 10000 # En sık kullanılan 10.000 kelimeyi kullan
tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(df['clean_review'])

X = tokenizer.texts_to_sequences(df['clean_review'])
y = df['sentiment_encoded'].values

print(f"Örnek bir metin sırası: {X[0][:20]}...")

maxlen = 200
X = pad_sequences(X, maxlen=maxlen, padding='post', truncating='post')

print(f"Pading uygulanmış örnek bir metin sırası: {X[0][:20]}...")
print(f"X'in şekli: {X.shape}")
print(f"y'nin şekli: {y.shape}")


### 2. Veriyi Eğitim ve Test Setlerine Ayırma

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

print(f"Eğitim seti boyutu: {X_train.shape[0]}")
print(f"Doğrulama seti boyutu: {X_val.shape[0]}")
print(f"Test seti boyutu: {X_test.shape[0]}")

### 3. RNN Modeli Oluşturma

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

embedding_dim = 128

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=maxlen),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Model özeti:")
model.summary()


### 4. Modeli Eğitme

In [ ]:
epochs = 5
batch_size = 64

history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val)
)


### 5. Model Değerlendirme

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Eğitim Doğruluğu')
plt.plot(history.history['val_accuracy'], label='Doğrulama Doğruluğu')
plt.title('Doğruluk Oranı (Accuracy)')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Eğitim Kaybı')
plt.plot(history.history['val_loss'], label='Doğrulama Kaybı')
plt.title('Kayıp Fonksiyonu (Loss)')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()

plt.tight_layout()
plt.show()

# Test seti üzerinde değerlendirme
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Seti Kaybı: {loss:.4f}")
print(f"Test Seti Doğruluğu: {accuracy:.4f}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

print("\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print("\nKarmaşıklık Matrisi (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred))

# Örnek tahminler yapalım
def predict_sentiment(text):
    clean = clean_text(text)
    seq = tokenizer.texts_to_sequences([clean])
    padded = pad_sequences(seq, maxlen=maxlen, padding='post', truncating='post')
    prediction = model.predict(padded)[0][0]
    sentiment = label_encoder.inverse_transform([int(round(prediction))])[0]
    return sentiment, prediction

print("\nÖrnek Tahminler:")
review1 = "This movie was absolutely fantastic! I loved every minute of it."
sentiment1, prob1 = predict_sentiment(review1)
print(f"'{review1}' -> Tahmin: {sentiment1} (Olasılık: {prob1:.4f})")

review2 = "What a terrible film. I wasted my money and time."
sentiment2, prob2 = predict_sentiment(review2)
print(f"'{review2}' -> Tahmin: {sentiment2} (Olasılık: {prob2:.4f})")

review3 = "It was an okay movie, nothing special but not bad either."
sentiment3, prob3 = predict_sentiment(review3)
print(f"'{review3}' -> Tahmin: {sentiment3} (Olasılık: {prob3:.4f})")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Karmaşıklık Matrisi (Confusion Matrix)')
plt.xlabel('Tahmin Edilen Etiket')
plt.ylabel('Gerçek Etiket')
plt.show()


### Veri Seti için WordCloud Görselleştirmesi

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

all_reviews_text = " ".join(df['clean_review'].tolist())

# WordCloud oluşturma
wordcloud_all = WordCloud(width=800, height=400, background_color='white').generate(all_reviews_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_all, interpolation='bilinear')
plt.axis('off')
plt.title('Tüm Veri Setindeki En Sık Kullanılan Kelimeler (WordCloud)')
plt.show()
